In [ ]:
import pandas as pd
import polars as pl
import plotly.express as px
from lstm_translator.tokenizer import BPETokenizer

In [2]:
def load_data_csv(filename) -> tuple[list, list]:
    data = pd.read_csv(filename).dropna()
    return data['en'].astype(str).to_list(), data['fr'].astype(str).to_list()

In [3]:
en_tokenizer = BPETokenizer.load('./en_bpe_model.json')
fr_tokenizer = BPETokenizer.load('./fr_bpe_model.json')

In [4]:
en_texts, fr_texts = load_data_csv('./en-fr-7-len-317-sampled.csv')

In [5]:
en_tokenized = en_tokenizer.encode_batch(en_texts)
fr_tokenized = fr_tokenizer.encode_batch(fr_texts)

In [6]:
df = pl.DataFrame({
    'en_tokenized': en_tokenized,
    'fr_tokenized': fr_tokenized
}) 

In [7]:
df

en_tokenized,fr_tokenized
list[str],list[str]
"[""Evaluation</w>"", ""H"", … ""or</w>""]","[""E"", ""valuation</w>"", … ""teur</w>""]"
"[""Compl"", ""iance</w>"", … ""17</w>""]","[""Confor"", ""mité</w>"", … ""17</w>""]"
"[""F"", ""igh"", … ""1</w>""]","[""Com"", ""bat"", … ""1</w>""]"
"[""P"", ""ati"", … ""Organization</w>""]","[""Organis"", ""me</w>"", … ""patients</w>""]"
"[""Den"", ""mark</w>"", … ""Institute</w>""]","[""An"", ""gl"", … ""A</w>""]"
…,…
"[""Trade</w>"", ""shows</w>"", … ""itions</w>""]","[""National</w>"", ""Association</w>"", … ""ors</w>""]"
"[""En"", ""dang"", … ""Canada</w>""]","[""Es"", ""p"", … ""Canada</w>""]"
"[""F"", ""uture</w>"", … ""treaty</w>""]","[""Les</w>"", ""élarg"", … ""traité</w>""]"


In [9]:
df = df.with_columns(
    pl.col('en_tokenized').list.len().alias('en_token_count'),
    pl.col('fr_tokenized').list.len().alias('fr_token_count')
)

In [13]:
fig = px.histogram(x=df['en_token_count'], nbins=20, title='English Token Count Distribution', marginal='box')  
fig.show()

In [26]:
long_sents= df.filter(
    pl.col('en_tokenized').list.len() > 50,
)

1912

In [ ]:
import torch

def set_max_split_size_mb(model, max_split_size_mb):
    """
    Set the max_split_size_mb parameter in PyTorch to avoid fragmentation.
    
    Args:
        model (torch.nn.Module): The PyTorch model.
        max_split_size_mb (int): The desired value for max_split_size_mb in megabytes.
    """
    for param in model.parameters():
        param.requires_grad = False  # Disable gradient calculation to prevent unnecessary memory allocations

    # Dummy forward pass to initialize the memory allocator
    dummy_input = torch.randn(5, 10)
    model(dummy_input)

    # Get the current memory allocator state
    allocator = torch.cuda.memory

    # Update max_split_size_mb in the memory allocator
    allocator.set_max_split_size(max_split_size_mb * 1024 * 1024)

    for param in model.parameters():
        param.requires_grad = True  # Re-enable gradient calculation for training

# Example usage
if __name__ == "__main__":
    # Create your PyTorch model
    model = torch.nn.Linear(10, 5)

    # Set the desired max_split_size_mb value (e.g., 200 MB)
    max_split_size_mb = 200

    # Call the function to set max_split_size_mb
    set_max_split_size_mb(model, max_split_size_mb)

AttributeError: module 'torch.cuda.memory' has no attribute '_get_memory_allocator'